# Deepfake Face Swap

This notebook drives the `deepfake_faceswap` package: it downloads the
`inswapper_128` face-swap model, swaps a source face onto a target video using
[insightface](https://github.com/deepinsight/insightface) directly, and
(optionally) renders a side-by-side comparison video and a GIF preview.

Works both locally and on Google Colab.

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Make the project package importable when this notebook runs from notebooks/.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from deepfake_faceswap import (
    download_face_swap_model,
    resolve_demo_paths,
    run_face_swap,
    combine_frames_with_video,
    save_video_as_gif,
)

## Step 1: Set up the environment
Download the face-swap model (`insightface` itself is installed like any other
dependency via `requirements.txt`).

In [ ]:
model_path = download_face_swap_model(PROJECT_ROOT / "models")

## Step 2 (Colab only): mount Google Drive

Skip this cell when running locally — local paths already point at `assets/`.

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

## Step 3: Configure input/output paths

Pick a demo by name — `resolve_demo_paths` resolves the matching video/face
under `assets/` and the output paths under `output/<demo>/`, the same lookup
the CLI's `demo` subcommand uses, so both demos run through one code path.

In [ ]:
DEMO_NAME = "demo_1"  # or "demo_2" — switch to run the other bundled demo

demo_paths = resolve_demo_paths(PROJECT_ROOT, DEMO_NAME)
target_video_path = demo_paths.target_video
source_face_image_path = demo_paths.source_face
output_video_path = demo_paths.output_video

# Example Colab override (bypasses resolve_demo_paths — point these at Drive instead):
# target_video_path = "/content/drive/MyDrive/content/InputVideos/video.mp4"
# source_face_image_path = "/content/drive/MyDrive/content/Faces/elonMusk.jpeg"
# output_video_path = "/content/drive/MyDrive/content/Output/output.mp4"

## Step 4: Run the face swap

In [ ]:
run_face_swap(
    target_video_path=target_video_path,
    source_face_image_path=source_face_image_path,
    output_video_path=output_video_path,
    model_path=model_path,
    execution_provider="cpu",  # switch to "cuda" if a GPU is available (e.g. on Colab)
)

## Step 5 (optional): render a side-by-side comparison video

`run_face_swap` writes one swapped frame per video frame under
`output/frames/<video_name>/` by default. Point `frames_dir` at that folder to
compare it against the source video.

In [ ]:
frames_dir = output_video_path.parent / "frames" / target_video_path.stem
combined_video_path = demo_paths.combined_video

combine_frames_with_video(
    frames_dir=frames_dir,
    source_video_path=target_video_path,
    output_video_path=combined_video_path,
)

## Step 6 (optional): render a GIF preview

Turns the comparison video into a lightweight, inline-renderable GIF — handy
for embedding a preview in a README.

In [ ]:
gif_path = demo_paths.gif

save_video_as_gif(
    video_path=combined_video_path,
    gif_path=gif_path,
    fps=10,
    width=480,
)